# 🧮 Geometries for swimming embryo chimeras
This notebook provides tools for translating "organism-based" parameters into explicit geometric parameters for the [ChimeraDesign](./ChimeraDesign.ipynb) notebook, for simulation in the [ChimeraSwim](./ChimeraSwim.ipynb) model.

## Organism-based *vs.* geometrical parameters
The motivation for defining two sets of parameters is that they define larval morphologies in ways that lend themselves to two different purposes:
1. Organism-based parameters are intuitive to use in posing biological hypotheses, because they are expressed in terms of organismal constraints such as the tissue volume, and measurable functional traits such as excess density.
2. Geometrical parameters are necessary for constructing elements of the biomechanical model, but are unintuitive for posing biological hypotheses because they confound variations in multiple organismal traits.

Specifying either of these parameter sets also specifies the other.
Because the translations between them are not straightforward, they are implemeted in code below.

### Geometrical parameters
Larval morphologies in the [ChimeraSwim](./ChimeraSwim.ipynb) are defined by the geometrical parameters shown in this diagram:

![alt text](ChimeraSpheroid_geometry2.png "Surface shape parameters")

The parameters defining the external shape of the larva are:
- $D_s$: the maximum diameter of the external surface, at its equator
- $L_{s_1}$: the length of the top semi-spheroid (distance from the upper tip to the equator) of the external surface
- $L_{s_2}$: the length of the bottom semi-spheroid (distance from the lower tip to the equator) of the external surface

The parameters defining the shape of the internal "inclusion" are:
- $D_i$: the maximum diameter of the inclusion, at its equator
- $L_{i_1}$: the length of the top semi-spheroid (distance from the upper tip to the equator) of the inclusion
- $L_{i_2}$: the length of the bottom semi-spheroid (distance from the lower tip to the equator) of the inclusion
- $h_i$: the height of the inclusion equator above the external shape equator

It's assumed that the ambient seawater density is $1030 \frac{kg}{m^3}$.

### Organism-based parameters


In [1]:
%matplotlib widget
from matplotlib import pyplot
pyplot.ioff()
# Import modules
from math import pi
# Import widget infrastructure
from ipywidgets import interact, interactive, fixed, interact_manual, Output
import ipywidgets as widgets

In [2]:
def get_geom_pars(Vt,alpha,eta,rho_tissue,rho_incl,rho_excess,rho_seawater=1030,sigma = 0.8):
    drho_tissue = rho_tissue - rho_seawater
    drho_incl = rho_incl - rho_seawater
    # calculate geometry of tissue-only chimera
    Dt = (6*Vt/(pi*alpha))**(1/3)
    Lt0 = alpha*Dt
    Lt2 = eta * Lt0
    Lt1 = Lt0 - Lt2
    # Compute "inflated surface after inclusion to adjust excess density
    beta = (drho_tissue-drho_incl)/(rho_excess-drho_incl)
    Ds = beta**(1/3) * Dt
    Vs = beta*Vt
    Ds =  (6*Vs/(pi*alpha))**(1/3)
    Ls0 = alpha*Ds
    Ls2 = eta * Ls0
    Ls1 = Ls0 - Ls2
    
    Vi = Vs - Vt
    Di =  (6*Vi/(pi*alpha))**(1/3)
    Li0 = alpha*Ds
    Li2 = eta * Li0
    Li1 = Li0 - Li2
    
    xsi = (1-eta)*(sigma - ((beta-1)/beta)**(1/3))
    hi = xsi * Li0

    # Print results
    print('External surface parameters:')
    print(f'Ds = {Ds:.2e}')
    print(f'Ls0 = {Ls0:.2e}')
    print(f'Ls1 = {Ls1:.2e}')
    print(f'Ls2 = {Ls2:.2e}')
    
    print('\nInclusion parameters:')
    print(f'Di = {Di:.2e}')
    print(f'Li0 = {Li0:.2e}')
    print(f'Li1 = {Li1:.2e}')
    print(f'Li2 = {Li2:.2e}')
    print(f'hi = {hi:.2e}')


In [3]:
# Set starting organismal parameters
_Vt=widgets.FloatText(value=(5.e-5)**3,width=10,description = r"$V_t$ ($m^3$)")
_alpha=widgets.FloatText(value=2.,description = r"$\alpha$")
_eta=widgets.FloatText(value=0.75,description = r"$\eta$")
_rho_tissue=widgets.FloatText(value=1070,description = r"$\rho_{tissue}$ ($\frac{kg}{m^3}$)")
_rho_incl=widgets.FloatText(value=1030,description = r"$\rho_{incl}$ ($\frac{kg}{m^3}$)")
_rho_excess=widgets.FloatText(value=25,description = r"$\rho_{excess}$ ($\frac{kg}{m^3}$)")

ui0s = widgets.VBox([_Vt,_alpha,_eta])
ui1s = widgets.VBox([_rho_tissue,_rho_incl,_rho_excess])
ui01s = widgets.HBox([ui0s,ui1s])


outs = widgets.interactive_output(get_geom_pars,{'Vt':_Vt,'alpha':_alpha,'eta':_eta,
                                                'rho_tissue':_rho_tissue,
                                                'rho_incl':_rho_incl,
                                                'rho_excess':_rho_excess})
display(ui01s,outs)

Output()

:::{figure} #cd_1
:placeholder: ./images/CG_1.png
:align: left
:::